# Procesos estocásticos, ruido blanco, PSD y dinámica de Langevin
## Tutorial computacional basado en las secciones de sistemas estocásticos de *KalmanFilters_1.pdf*

Este cuaderno desarrolla cinco ideas que conducen directamente al filtrado de Kalman:

1. procesos aleatorios (estocásticos) vectoriales;
2. ruido blanco;
3. densidad espectral de potencia;
4. sistemas dinámicos en tiempo discreto con entradas aleatorias;
5. ecuaciones diferenciales estocásticas y la ecuación de Langevin, incluida la evolución temporal de la función de densidad de probabilidad.

El código utiliza ensambles de Monte Carlo para relacionar trayectorias aleatorias individuales con medias, matrices de covarianza, funciones de autocorrelación, espectros y densidades de probabilidad.


## Mapa conceptual

| Objeto | Dominio | Qué describe |
|---|---|---|
| Realización $x_k(\omega)$ | Tiempo | Una trayectoria posible |
| Media $m_k=\mathbb E[X_k]$ | Tiempo | Centro del ensamble en cada instante |
| Covarianza $C_{k,\ell}$ | Dos índices temporales | Dependencia entre distintos tiempos o componentes |
| Autocovarianza $R_X[r]$ | Retardo temporal | Dependencia en función de la separación |
| PSD $S_X(f)$ | Frecuencia | Distribución de la potencia media sobre la frecuencia |
| Covarianza del estado $P_k$ | Tiempo | Incertidumbre propagada por el modelo dinámico |
| PDF $p(x,t)$ | Estado y tiempo | Distribución de un ensamble conforme evoluciona |

> Una trayectoria, una PDF y una PSD son perspectivas distintas del mismo fenómeno estocástico. Una sola realización no revela por sí misma la ley de probabilidad completa.


## Preparación del entorno

Solo se requieren NumPy y Plotly:

```bash
pip install numpy plotly
```


In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(seed=2026)
integrate = getattr(np, 'trapezoid', np.trapz)


def normal_pdf(x, mean, std):
    """Evalúa una PDF gaussiana escalar; std puede admitir broadcasting."""
    x = np.asarray(x, dtype=float)
    std = np.asarray(std, dtype=float)
    if np.any(std <= 0):
        raise ValueError('Todas las desviaciones estándar deben ser positivas.')
    return np.exp(-0.5 * ((x - mean) / std) ** 2) / (np.sqrt(2 * np.pi) * std)


# 1. Procesos aleatorios (estocásticos) vectoriales

Un proceso estocástico vectorial es una familia de vectores aleatorios indexados por el tiempo:

$$\{X_k:k=0,1,2,\ldots\},\qquad X_k\in\mathbb R^n.
$$

Existen dos perspectivas complementarias:

- Si se fija un resultado $\omega$, $x_0(\omega),x_1(\omega),\ldots$ es una **trayectoria muestral** o realización.
- Si se fija un instante $k$, $X_k$ es un vector aleatorio descrito por una distribución sobre un **ensamble** de sistemas posibles.

Los dos primeros momentos son

$$m_k=\mathbb E[X_k],$$

y

$$C_{k,\ell}=\mathbb E[(X_k-m_k)(X_\ell-m_\ell)^\top].$$

Un proceso es estacionario en sentido amplio cuando su media es constante y su covarianza depende únicamente del retardo $r=k-\ell$. La estacionariedad estricta es más fuerte: todas las distribuciones de dimensión finita deben permanecer invariantes ante un desplazamiento temporal común.

## Ejemplo: recursión estocástica estable de dos dimensiones

Simulamos

$$X_{k+1}=FX_k+W_k,\qquad W_k\sim\mathcal N(0,Q).$$

La condición inicial determinista hace que el proceso sea inicialmente no estacionario. Como $F$ es estable, su media decae y su covarianza se aproxima a un valor estacionario.


In [ ]:
F_vec = np.array([[0.92, 0.18],
                  [-0.08, 0.85]])
Q_vec = np.array([[0.08, 0.02],
                  [0.02, 0.05]])
x0_vec = np.array([2.0, -1.0])
n_steps_vec = 160
n_ensemble_vec = 2_000
L_Q_vec = np.linalg.cholesky(Q_vec)

X_vec = np.empty((n_ensemble_vec, 2, n_steps_vec))
X_vec[:, :, 0] = x0_vec
for k in range(n_steps_vec - 1):
    noise_k = rng.standard_normal((n_ensemble_vec, 2)) @ L_Q_vec.T
    X_vec[:, :, k + 1] = X_vec[:, :, k] @ F_vec.T + noise_k

mean_vec = X_vec.mean(axis=0)
var_vec = X_vec.var(axis=0, ddof=1)
time_vec = np.arange(n_steps_vec)

# Iteración de punto fijo para la covarianza estacionaria P = F P F^T + Q.
P_stationary = np.zeros((2, 2))
for _ in range(10_000):
    P_next = F_vec @ P_stationary @ F_vec.T + Q_vec
    if np.allclose(P_next, P_stationary, atol=1e-13, rtol=0.0):
        break
    P_stationary = P_next

print('Autovalores de F:', np.linalg.eigvals(F_vec))
print('Media empírica final:', mean_vec[:, -1])
print('Covarianza estacionaria:\n', P_stationary)
print('Covarianza empírica final:\n', np.cov(X_vec[:, :, -1], rowvar=False))

assert np.all(np.abs(np.linalg.eigvals(F_vec)) < 1.0)
assert np.allclose(np.cov(X_vec[:, :, -1], rowvar=False), P_stationary, atol=0.08)

fig_process = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Trayectorias muestrales de X₁', 'Trayectorias muestrales de X₂',
                    'Medias del ensamble', 'Varianzas del ensamble')
)
for j in range(18):
    fig_process.add_trace(
        go.Scatter(x=time_vec, y=X_vec[j, 0], mode='lines',
                   line=dict(width=1), opacity=0.35, showlegend=False),
        row=1, col=1
    )
    fig_process.add_trace(
        go.Scatter(x=time_vec, y=X_vec[j, 1], mode='lines',
                   line=dict(width=1), opacity=0.35, showlegend=False),
        row=1, col=2
    )

fig_process.add_trace(go.Scatter(x=time_vec, y=mean_vec[0], mode='lines',
                                 line=dict(width=3), name='E[X₁]'), row=2, col=1)
fig_process.add_trace(go.Scatter(x=time_vec, y=mean_vec[1], mode='lines',
                                 line=dict(width=3), name='E[X₂]'), row=2, col=1)
fig_process.add_trace(go.Scatter(x=time_vec, y=var_vec[0], mode='lines',
                                 line=dict(width=3), name='Var(X₁)'), row=2, col=2)
fig_process.add_trace(go.Scatter(x=time_vec, y=var_vec[1], mode='lines',
                                 line=dict(width=3), name='Var(X₂)'), row=2, col=2)
fig_process.add_hline(y=P_stationary[0, 0], line_dash='dash', line_color='royalblue',
                      row=2, col=2)
fig_process.add_hline(y=P_stationary[1, 1], line_dash='dash', line_color='darkorange',
                      row=2, col=2)
fig_process.update_xaxes(title_text='Paso temporal k')
fig_process.update_layout(
    title='Proceso estocástico vectorial: trayectorias y estadísticas del ensamble',
    template='plotly_white', width=1_050, height=760
)
fig_process.show()


### Interpretación del resultado

Cada curva delgada es una realización válida. La media del ensamble se calcula verticalmente sobre muchas realizaciones en un instante fijo, no promediando una trayectoria a lo largo del tiempo. Esos dos promedios solo coinciden bajo condiciones adicionales, como la ergodicidad.

La dinámica estable pierde gradualmente la información de la condición inicial determinista. La media se aproxima a cero y la covarianza se aproxima a la solución de la ecuación de Lyapunov discreta

$$P_\infty=F P_\infty F^\top+Q.
$$


# 2. Ruido blanco

Un proceso vectorial de ruido blanco en tiempo discreto y media cero satisface

$$\mathbb E[W_k]=0,$$

y

$$\mathbb E[W_kW_\ell^\top]=Q\,\delta_{k\ell},$$

donde $\delta_{k\ell}=1$ si $k=\ell$ y 0 en otro caso. Por tanto, las muestras de instantes diferentes están incorrelacionadas.

Distinciones importantes:

- *Blanco* describe la estructura temporal de segundo orden; no requiere una distribución gaussiana de amplitudes.
- El *ruido blanco gaussiano* tiene muestras conjuntamente gaussianas. En el caso gaussiano, correlación cruzada nula implica independencia.
- Las componentes de un mismo instante todavía pueden estar correlacionadas cuando $Q$ no es diagonal.
- El ruido blanco ideal tiene ancho de banda infinito y es una abstracción; los sensores físicos muestreados siempre tienen ancho de banda limitado.

Estimamos la autocovarianza escalar

$$\hat R[r]=\frac{1}{N}\sum_{k=0}^{N-r-1}(w_k-\bar w)(w_{k+r}-\bar w).$$


In [ ]:
def biased_autocovariance(signal, max_lag):
    signal = np.asarray(signal, dtype=float)
    centered = signal - signal.mean()
    N = centered.size
    return np.array([
        centered[:N - lag] @ centered[lag:] / N
        for lag in range(max_lag + 1)
    ])

N_white = 8_192
Q_white = 1.5
w = np.sqrt(Q_white) * rng.standard_normal(N_white)
max_lag = 50
R_hat = biased_autocovariance(w, max_lag)
rho_hat = R_hat / R_hat[0]
lags = np.arange(max_lag + 1)
approx_bound = 1.96 / np.sqrt(N_white)

print(f'Media muestral: {w.mean():.5f}')
print(f'Varianza muestral: {w.var(ddof=1):.5f}; valor objetivo Q={Q_white:.5f}')
print(f'Mayor |ACF muestral| fuera del retardo 0: {np.max(np.abs(rho_hat[1:])):.4f}')
print(f'Límite de referencia aproximado del 95 %: ±{approx_bound:.4f}')

fig_white = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Una realización', 'Autocorrelación estimada')
)
fig_white.add_trace(
    go.Scatter(x=np.arange(350), y=w[:350], mode='lines', name='w[k]'),
    row=1, col=1
)
fig_white.add_trace(
    go.Bar(x=lags, y=rho_hat, name='ACF muestral'),
    row=1, col=2
)
fig_white.add_hline(y=approx_bound, line_dash='dash', line_color='red', row=1, col=2)
fig_white.add_hline(y=-approx_bound, line_dash='dash', line_color='red', row=1, col=2)
fig_white.update_xaxes(title_text='Paso temporal k', row=1, col=1)
fig_white.update_xaxes(title_text='Retardo r', row=1, col=2)
fig_white.update_yaxes(title_text='Amplitud', row=1, col=1)
fig_white.update_yaxes(title_text='Correlación', row=1, col=2)
fig_white.update_layout(
    title='Ruido blanco gaussiano en el dominio temporal',
    template='plotly_white', width=1_020, height=470, showlegend=False
)
fig_white.show()


# 3. Densidad espectral de potencia

Para un proceso discreto de media cero y estacionario en sentido amplio, la relación de Wiener–Khinchin conecta la autocovarianza con la densidad espectral de potencia:

$$S_X(f)=\sum_{r=-\infty}^{\infty}R_X[r]e^{-j2\pi fr},$$

con la transformada inversa

$$R_X[r]=\int_{-1/2}^{1/2}S_X(f)e^{j2\pi fr}\,df.
$$

Aquí, $f$ se mide en ciclos por muestra. En particular,

$$R_X[0]=\operatorname{Var}(X_k)=\int_{-1/2}^{1/2}S_X(f)\,df.
$$

Para el ruido blanco, $R_W[r]=Q\delta[r]$ y, por ello, $S_W(f)=Q$ es plana. Hacer pasar ruido blanco por una dinámica produce ruido coloreado. Para el sistema escalar AR(1)

$$Y_k=aY_{k-1}+W_k,$$

la PSD es

$$S_Y(f)=\frac{Q}{|1-ae^{-j2\pi f}|^2}.
$$

Promediamos periodogramas sobre un ensamble para reducir la elevada varianza de un periodograma individual.


In [ ]:
def ensemble_periodogram(signals):
    """Periodograma bilateral promediado sobre un ensamble, en ciclos/muestra."""
    signals = np.asarray(signals, dtype=float)
    N = signals.shape[-1]
    centered = signals - signals.mean(axis=-1, keepdims=True)
    spectrum = np.fft.fft(centered, axis=-1)
    psd = np.mean(np.abs(spectrum) ** 2 / N, axis=0)
    frequency = np.fft.fftfreq(N, d=1.0)
    return np.fft.fftshift(frequency), np.fft.fftshift(psd)

n_records = 300
N_psd = 1_024
Q_psd = 1.0
a_ar = 0.92

white_records = np.sqrt(Q_psd) * rng.standard_normal((n_records, N_psd))
colored_records = np.empty_like(white_records)
colored_records[:, 0] = rng.normal(
    0.0, np.sqrt(Q_psd / (1.0 - a_ar ** 2)), size=n_records
)
for k in range(1, N_psd):
    colored_records[:, k] = a_ar * colored_records[:, k - 1] + white_records[:, k]

f_psd, S_white_hat = ensemble_periodogram(white_records)
_, S_colored_hat = ensemble_periodogram(colored_records)
S_white_theory = np.full_like(f_psd, Q_psd)
S_colored_theory = Q_psd / np.abs(1.0 - a_ar * np.exp(-1j * 2.0 * np.pi * f_psd)) ** 2
df = 1.0 / N_psd

print(f'Potencia del ruido blanco obtenida de la PSD: {np.sum(S_white_hat) * df:.4f}')
print(f'Varianza objetivo del ruido blanco:           {Q_psd:.4f}')
print(f'Potencia del AR(1) obtenida de la PSD:         {np.sum(S_colored_hat) * df:.4f}')
print(f'Varianza teórica del AR(1):                    {Q_psd / (1-a_ar**2):.4f}')

assert np.isclose(np.sum(S_white_hat) * df, Q_psd, rtol=0.03)
assert np.isclose(np.sum(S_colored_hat) * df, Q_psd / (1-a_ar**2), rtol=0.08)

fig_psd = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Ruido blanco: espectro plano', 'AR(1): espectro coloreado pasa-bajas')
)
fig_psd.add_trace(go.Scatter(x=f_psd, y=S_white_hat, mode='lines',
                             name='PSD blanca estimada'), row=1, col=1)
fig_psd.add_trace(go.Scatter(x=f_psd, y=S_white_theory, mode='lines',
                             line=dict(color='black', dash='dash', width=3),
                             name='Teoría para ruido blanco'), row=1, col=1)
fig_psd.add_trace(go.Scatter(x=f_psd, y=S_colored_hat, mode='lines',
                             name='PSD AR(1) estimada'), row=1, col=2)
fig_psd.add_trace(go.Scatter(x=f_psd, y=S_colored_theory.real, mode='lines',
                             line=dict(color='black', dash='dash', width=3),
                             name='Teoría AR(1)'), row=1, col=2)
fig_psd.update_xaxes(title_text='Frecuencia [ciclos/muestra]')
fig_psd.update_yaxes(title_text='PSD')
fig_psd.update_layout(
    title='Densidad espectral de potencia: ruido blanco frente a ruido coloreado por la dinámica',
    template='plotly_white', width=1_050, height=500
)
fig_psd.show()


# 4. Sistemas dinámicos en tiempo discreto con entradas aleatorias

Consideremos el modelo estocástico lineal en espacio de estados

$$X_{k+1}=F_kX_k+B_ku_k+G_kW_k,$$

$$Z_k=H_kX_k+V_k,$$

donde

$$W_k\sim\mathcal N(0,Q_k),\qquad V_k\sim\mathcal N(0,R_k).$$

Suponiendo que $W_k$ es independiente de $X_k$, la media y la covarianza del estado se propagan como

$$m_{k+1}=F_km_k+B_ku_k,$$

$$P_{k+1}=F_kP_kF_k^\top+G_kQ_kG_k^\top.
$$

Estas son las **ecuaciones de predicción** del filtro de Kalman antes de la actualización con una medición. La diferencia fundamental respecto a una simulación determinista es que propagamos una distribución, resumida aquí mediante sus dos primeros momentos.

## Ejemplo: movimiento a velocidad constante excitado por aceleración aleatoria

Con $X_k=[p_k,v_k]^\top$ y un intervalo de muestreo $\Delta t$,

$$
F=\begin{bmatrix}1&\Delta t\\0&1\end{bmatrix},\qquad
G=\begin{bmatrix}\Delta t^2/2\\\Delta t\end{bmatrix}.
$$

Una aceleración aleatoria escalar $W_k$ afecta la posición y la velocidad a través de $G$. Las mediciones de posición están corrompidas por $V_k$.


In [ ]:
dt_dyn = 0.1
F_dyn = np.array([[1.0, dt_dyn],
                  [0.0, 1.0]])
G_dyn = np.array([[0.5 * dt_dyn ** 2],
                  [dt_dyn]])
H_dyn = np.array([[1.0, 0.0]])
q_acceleration = 0.8
R_measurement = 4.0
m0_dyn = np.array([0.0, 5.0])
P0_dyn = np.diag([1.0, 0.25])
n_steps_dyn = 151
n_ensemble_dyn = 5_000
time_dyn = np.arange(n_steps_dyn) * dt_dyn

# Ensamble de Monte Carlo.
states = m0_dyn + rng.standard_normal((n_ensemble_dyn, 2)) @ np.linalg.cholesky(P0_dyn).T
state_history_display = np.empty((25, 2, n_steps_dyn))
state_history_display[:, :, 0] = states[:25]
empirical_mean = np.empty((2, n_steps_dyn))
empirical_cov = np.empty((2, 2, n_steps_dyn))
empirical_mean[:, 0] = states.mean(axis=0)
empirical_cov[:, :, 0] = np.cov(states, rowvar=False)

for k in range(n_steps_dyn - 1):
    random_acceleration = np.sqrt(q_acceleration) * rng.standard_normal(n_ensemble_dyn)
    states = states @ F_dyn.T + random_acceleration[:, None] * G_dyn.ravel()
    state_history_display[:, :, k + 1] = states[:25]
    empirical_mean[:, k + 1] = states.mean(axis=0)
    empirical_cov[:, :, k + 1] = np.cov(states, rowvar=False)

# Propagación analítica de los momentos.
mean_pred = np.empty((2, n_steps_dyn))
P_pred = np.empty((2, 2, n_steps_dyn))
mean_pred[:, 0] = m0_dyn
P_pred[:, :, 0] = P0_dyn
Q_state = (G_dyn * q_acceleration) @ G_dyn.T
for k in range(n_steps_dyn - 1):
    mean_pred[:, k + 1] = F_dyn @ mean_pred[:, k]
    P_pred[:, :, k + 1] = F_dyn @ P_pred[:, :, k] @ F_dyn.T + Q_state

# Una secuencia de mediciones ruidosas, mostrada únicamente para su interpretación.
true_position = state_history_display[0, 0]
measurements = true_position + np.sqrt(R_measurement) * rng.standard_normal(n_steps_dyn)
sigma_position = np.sqrt(P_pred[0, 0])
upper_position = mean_pred[0] + 2.0 * sigma_position
lower_position = mean_pred[0] - 2.0 * sigma_position

final_position_var_error = abs(empirical_cov[0, 0, -1] - P_pred[0, 0, -1]) / P_pred[0, 0, -1]
final_velocity_var_error = abs(empirical_cov[1, 1, -1] - P_pred[1, 1, -1]) / P_pred[1, 1, -1]
print('Media predicha final:', mean_pred[:, -1])
print('Media empírica final:', empirical_mean[:, -1])
print('Covarianza predicha final:\n', P_pred[:, :, -1])
print('Covarianza empírica final:\n', empirical_cov[:, :, -1])
print(f'Error relativo de la varianza de posición: {final_position_var_error:.2%}')
print(f'Error relativo de la varianza de velocidad: {final_velocity_var_error:.2%}')

assert final_position_var_error < 0.06
assert final_velocity_var_error < 0.06

fig_dyn = make_subplots(
    rows=2, cols=2, specs=[[{'colspan': 2}, None], [{}, {}]],
    subplot_titles=('Una realización, mediciones e incertidumbre predicha',
                    'Varianza de posición', 'Varianza de velocidad')
)
fig_dyn.add_trace(go.Scatter(x=time_dyn, y=upper_position, mode='lines',
                             line=dict(width=0), showlegend=False), row=1, col=1)
fig_dyn.add_trace(go.Scatter(x=time_dyn, y=lower_position, mode='lines',
                             fill='tonexty', fillcolor='rgba(65,105,225,0.18)',
                             line=dict(width=0), name='Predicción ±2σ'), row=1, col=1)
fig_dyn.add_trace(go.Scatter(x=time_dyn, y=mean_pred[0], mode='lines',
                             line=dict(color='royalblue', width=3),
                             name='Media predicha'), row=1, col=1)
fig_dyn.add_trace(go.Scatter(x=time_dyn, y=true_position, mode='lines',
                             line=dict(color='black', width=2),
                             name='Una trayectoria verdadera'), row=1, col=1)
fig_dyn.add_trace(go.Scatter(x=time_dyn, y=measurements, mode='markers',
                             marker=dict(size=4, opacity=0.45, color='crimson'),
                             name='Mediciones ruidosas'), row=1, col=1)
fig_dyn.add_trace(go.Scatter(x=time_dyn, y=P_pred[0, 0], mode='lines',
                             line=dict(width=3), name='P₁₁ predicha'), row=2, col=1)
fig_dyn.add_trace(go.Scatter(x=time_dyn, y=empirical_cov[0, 0], mode='lines',
                             line=dict(width=2, dash='dash'), name='Var(p) empírica'), row=2, col=1)
fig_dyn.add_trace(go.Scatter(x=time_dyn, y=P_pred[1, 1], mode='lines',
                             line=dict(width=3), name='P₂₂ predicha'), row=2, col=2)
fig_dyn.add_trace(go.Scatter(x=time_dyn, y=empirical_cov[1, 1], mode='lines',
                             line=dict(width=2, dash='dash'), name='Var(v) empírica'), row=2, col=2)
fig_dyn.update_xaxes(title_text='Tiempo [s]')
fig_dyn.update_yaxes(title_text='Posición', row=1, col=1)
fig_dyn.update_yaxes(title_text='Varianza', row=2, col=1)
fig_dyn.update_yaxes(title_text='Varianza', row=2, col=2)
fig_dyn.update_layout(
    title='Las entradas aleatorias se propagan hacia la incertidumbre del estado',
    template='plotly_white', width=1_050, height=780
)
fig_dyn.show()


### Interpretación

La media sigue la trayectoria de velocidad constante sin ruido porque el ruido de proceso tiene media cero. La covarianza crece porque cada perturbación de aceleración añade incertidumbre y la dinámica transfiere a la posición la incertidumbre previa de la velocidad.

Las mediciones rojas se muestran, pero **no se utilizan** en esta simulación para corregir el estado. Por tanto, esto es una predicción estocástica y todavía no un filtro de Kalman completo. Una actualización de medición condicionaría la distribución del estado a cada medición observada y normalmente reduciría $P_k$.


# 5. Sistemas estocásticos y ecuación de Langevin

Un modelo de Langevin en tiempo continuo suele escribirse formalmente como

$$\frac{dX}{dt}=a(X,t)+b(X,t)\,\xi(t),$$

donde $\xi(t)$ es ruido blanco ideal. De manera más rigurosa, esta expresión representa la ecuación diferencial estocástica (SDE)

$$dX_t=a(X_t,t)dt+b(X_t,t)dW_t,$$

donde $W_t$ es un movimiento browniano. El ruido blanco se interpreta como la derivada generalizada del movimiento browniano; no se simula como una función puntual de valor finito. En un intervalo pequeño,

$$\Delta W_k\sim\mathcal N(0,\Delta t).$$

El método de Euler–Maruyama es

$$X_{k+1}=X_k+a(X_k,t_k)\Delta t+b(X_k,t_k)\sqrt{\Delta t}Z_k,
\qquad Z_k\sim\mathcal N(0,1).$$

## Ecuación de Langevin de Ornstein–Uhlenbeck

Utilizamos el modelo con reversión a la media

$$dX_t=-\lambda(X_t-\theta)dt+\sigma dW_t.
$$

La deriva atrae el estado hacia $\theta$, mientras que la difusión dispersa continuamente el ensamble. Partiendo de la condición determinista $X_0=x_0$, la distribución exacta es gaussiana:

$$m(t)=\theta+(x_0-\theta)e^{-\lambda t},$$

$$v(t)=\frac{\sigma^2}{2\lambda}\left(1-e^{-2\lambda t}\right).$$


In [ ]:
lambda_ou = 1.2
theta_ou = 0.0
sigma_ou = 0.8
x0_ou = 3.0
dt_ou = 0.01
T_ou = 5.0
n_steps_ou = int(T_ou / dt_ou)
n_paths_ou = 30_000
time_ou = np.arange(n_steps_ou + 1) * dt_ou

x_current = np.full(n_paths_ou, x0_ou)
n_display = 30
paths_ou = np.empty((n_display, n_steps_ou + 1))
paths_ou[:, 0] = x0_ou
snapshot_times = [0.25, 1.0, 3.0, 5.0]
snapshot_indices = {int(round(t / dt_ou)): t for t in snapshot_times}
snapshots_ou = {}

for k in range(n_steps_ou):
    dW = np.sqrt(dt_ou) * rng.standard_normal(n_paths_ou)
    x_current += -lambda_ou * (x_current - theta_ou) * dt_ou + sigma_ou * dW
    paths_ou[:, k + 1] = x_current[:n_display]
    if (k + 1) in snapshot_indices:
        snapshots_ou[snapshot_indices[k + 1]] = x_current.copy()

mean_ou = theta_ou + (x0_ou - theta_ou) * np.exp(-lambda_ou * time_ou)
var_ou = sigma_ou ** 2 / (2.0 * lambda_ou) * (1.0 - np.exp(-2.0 * lambda_ou * time_ou))
std_ou = np.sqrt(var_ou)

final_mean_error = abs(x_current.mean() - mean_ou[-1])
final_var_error = abs(x_current.var(ddof=1) - var_ou[-1])
print(f'Media teórica final: {mean_ou[-1]:.5f}')
print(f'Media final de Monte Carlo: {x_current.mean():.5f}')
print(f'Varianza teórica final: {var_ou[-1]:.5f}')
print(f'Varianza final de Monte Carlo: {x_current.var(ddof=1):.5f}')
print(f'Varianza estacionaria σ²/(2λ): {sigma_ou**2/(2*lambda_ou):.5f}')

assert final_mean_error < 0.02
assert final_var_error < 0.02

fig_ou_paths = go.Figure()
for j in range(n_display):
    fig_ou_paths.add_trace(go.Scatter(
        x=time_ou, y=paths_ou[j], mode='lines',
        line=dict(width=1), opacity=0.3, showlegend=False
    ))
fig_ou_paths.add_trace(go.Scatter(
    x=time_ou, y=mean_ou + 2.0 * std_ou, mode='lines',
    line=dict(width=0), showlegend=False
))
fig_ou_paths.add_trace(go.Scatter(
    x=time_ou, y=mean_ou - 2.0 * std_ou, mode='lines',
    fill='tonexty', fillcolor='rgba(65,105,225,0.18)',
    line=dict(width=0), name='Solución analítica ±2σ'
))
fig_ou_paths.add_trace(go.Scatter(
    x=time_ou, y=mean_ou, mode='lines',
    line=dict(color='black', width=4), name='Media analítica'
))
fig_ou_paths.update_layout(
    title='Trayectorias de Ornstein–Uhlenbeck y momentos analíticos',
    xaxis_title='Tiempo t', yaxis_title='Estado X(t)',
    template='plotly_white', width=950, height=560
)
fig_ou_paths.show()


## De la ecuación de Langevin a una PDF: ecuación de Fokker–Planck

La SDE describe trayectorias aleatorias. La densidad de probabilidad correspondiente satisface la ecuación de Fokker–Planck

$$
\frac{\partial p}{\partial t}
=-\frac{\partial}{\partial x}[a(x,t)p(x,t)]
+\frac{1}{2}\frac{\partial^2}{\partial x^2}[b^2(x,t)p(x,t)].
$$

Para el proceso de Ornstein–Uhlenbeck,

$$
\frac{\partial p}{\partial t}
=\lambda\frac{\partial}{\partial x}[(x-\theta)p]
+\frac{\sigma^2}{2}\frac{\partial^2p}{\partial x^2}.
$$

La siguiente superficie es la PDF analítica $p(x,t)$. Comienza concentrada cerca de $x_0$, su centro se desplaza hacia $\theta$ y su anchura se aproxima a la desviación estándar estacionaria $\sigma/\sqrt{2\lambda}$. La gráfica comienza en $t=\Delta t$ porque una condición inicial determinista es una delta de Dirac en $t=0$, no una densidad ordinaria de altura finita.


In [ ]:
t_pdf = np.linspace(dt_ou, T_ou, 220)
x_pdf = np.linspace(-2.5, 3.6, 500)
mean_pdf = theta_ou + (x0_ou - theta_ou) * np.exp(-lambda_ou * t_pdf)
var_pdf = sigma_ou ** 2 / (2.0 * lambda_ou) * (1.0 - np.exp(-2.0 * lambda_ou * t_pdf))
std_pdf = np.sqrt(var_pdf)
PXT = normal_pdf(x_pdf[None, :], mean_pdf[:, None], std_pdf[:, None])
normalization = integrate(PXT, x_pdf, axis=1)

print(f'Intervalo de normalización de la PDF en la malla graficada: '
      f'[{normalization.min():.6f}, {normalization.max():.6f}]')
assert np.allclose(normalization, 1.0, atol=2e-3)

fig_pdf_surface = go.Figure(go.Surface(
    x=x_pdf, y=t_pdf, z=PXT, colorscale='Viridis',
    colorbar=dict(title='p(x,t)')
))
fig_pdf_surface.update_layout(
    title='Evolución de la densidad de probabilidad de Ornstein–Uhlenbeck',
    scene=dict(
        xaxis_title='Estado x',
        yaxis_title='Tiempo t',
        zaxis_title='p(x,t)',
        camera=dict(eye=dict(x=1.45, y=-1.55, z=0.95))
    ),
    template='plotly_white', width=980, height=700
)
fig_pdf_surface.show()


## Validación de la PDF evolutiva mediante Monte Carlo

La PDF analítica describe el ensamble completo. Los histogramas obtenidos con un número finito de trayectorias de Euler–Maruyama deben aproximarse a ella. Las pequeñas diferencias restantes se deben al muestreo finito y al error de discretización temporal.


In [ ]:
fig_pdf_validation = make_subplots(
    rows=2, cols=2,
    subplot_titles=[f't = {t:g}' for t in snapshot_times]
)

for index, t_snapshot in enumerate(snapshot_times):
    row = index // 2 + 1
    col = index % 2 + 1
    samples_t = snapshots_ou[t_snapshot]
    mean_t = theta_ou + (x0_ou - theta_ou) * np.exp(-lambda_ou * t_snapshot)
    var_t = sigma_ou ** 2 / (2.0 * lambda_ou) * (1.0 - np.exp(-2.0 * lambda_ou * t_snapshot))
    grid_t = np.linspace(samples_t.min() - 0.3, samples_t.max() + 0.3, 500)

    fig_pdf_validation.add_trace(
        go.Histogram(
            x=samples_t, histnorm='probability density', nbinsx=90,
            opacity=0.55, showlegend=(index == 0), name='Histograma de Monte Carlo'
        ), row=row, col=col
    )
    fig_pdf_validation.add_trace(
        go.Scatter(
            x=grid_t, y=normal_pdf(grid_t, mean_t, np.sqrt(var_t)),
            mode='lines', line=dict(color='black', width=3),
            showlegend=(index == 0), name='PDF analítica'
        ), row=row, col=col
    )

fig_pdf_validation.update_xaxes(title_text='Estado x')
fig_pdf_validation.update_yaxes(title_text='Densidad')
fig_pdf_validation.update_layout(
    title='Ensambles de Euler–Maruyama frente a la PDF analítica',
    template='plotly_white', width=1_000, height=760, barmode='overlay'
)
fig_pdf_validation.show()


# Conexiones finales

Las cinco secciones forman una sola cadena conceptual:

1. Un proceso estocástico es una familia indexada de variables o vectores aleatorios.
2. El ruido blanco suministra innovaciones temporalmente incorrelacionadas.
3. La dinámica del sistema transforma su PSD plana en un espectro de estado coloreado.
4. En tiempo discreto, la dinámica propaga la media y la covarianza del estado.
5. En tiempo continuo, la SDE de Langevin propaga trayectorias aleatorias mientras que la ecuación de Fokker–Planck propaga su PDF.

Para sistemas lineales con condiciones iniciales gaussianas y ruido gaussiano, la distribución permanece gaussiana. Por tanto, dar seguimiento únicamente a la media y la covarianza es exacto. Esta propiedad de cerradura es fundamental para el filtro de Kalman clásico.

## Precauciones prácticas

- Las autocorrelaciones muestrales y los periodogramas nunca se ven perfectamente ideales con datos finitos.
- Una PSD plana no especifica la distribución de amplitudes.
- El ruido blanco continuo debe interpretarse mediante integrales estocásticas.
- Euler–Maruyama introduce error de discretización; disminuir $\Delta t$ mejora la aproximación, pero aumenta el costo.
- La recursión de covarianza supone que el ruido de proceso es independiente del estado actual, salvo que se añadan términos de covarianza cruzada.
- Los sistemas no lineales o no gaussianos generalmente requieren más que los dos primeros momentos.

## Ejercicios

1. Cambia los autovalores de `F_vec`. ¿Qué ocurre cuando uno de ellos sale del círculo unitario?
2. Genera ruido blanco uniforme con la misma varianza. Compara su ACF y su PSD con las del ruido blanco gaussiano.
3. Cambia `a_ar` de 0.92 a -0.92. Explica por qué el pico espectral se desplaza hacia la frecuencia de Nyquist.
4. Aumenta la varianza de medición `R_measurement`. ¿Qué cantidades graficadas cambian antes de implementar una actualización de Kalman?
5. Reduce `dt_ou` a la mitad y compara los errores finales de media y varianza de Euler–Maruyama.
6. Cambia `lambda_ou`, `theta_ou` y `sigma_ou`. Predice la media y la varianza estacionarias antes de ejecutar el código.
7. Sustituye la deriva OU por la deriva no lineal de doble pozo $a(x)=x-x^3$ y estima la PDF únicamente mediante histogramas de Monte Carlo.
